# Module 6 Homework

## Setup

In [ ]:
import os
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Force Java 21
os.environ["JAVA_HOME"] = "/usr/local/sdkman/candidates/java/21.0.9-ms"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe. 
Repartition the Dataframe to 4 partitions and save it to parquet.

In [ ]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

df_yellow = df_yellow.repartition(4)

df_yellow.write.parquet('data/pq/yellow/2025/11/', mode='overwrite')

!ls -lh data/pq/yellow/2025/11/

## Question 3: Count records

How many taxi trips were there on the 15th of November?
Consider only trips that started on the 15th of November.

In [ ]:
df_yellow \
    .withColumn('pickup_date', F.to_date(df_yellow.tpep_pickup_datetime)) \
    .filter(F.col('pickup_date') == '2025-11-15') \
    .count()

## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

In [ ]:
df_yellow \
    .withColumn('duration_hours', (
        F.col('tpep_dropoff_datetime').cast('timestamp').cast('long') - 
        F.col('tpep_pickup_datetime').cast('timestamp').cast('long')
    ) / 3600) \
    .select(F.max('duration_hours')) \
    .show()

## Question 6: Least frequent pickup location zone

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

**Logic Explanation:**
1. `Inner Join`: Connects the trip data with the zone metadata using the location IDs.
2. `groupBy('Zone')`: Groups the results by the human-readable zone name.
3. `count()`: Counts trips per zone.
4. `orderBy('count', ascending=True)`: Sorts to show the least busy zones first.

In [10]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [11]:
df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [12]:
df_yellow.join(df_zones, df_yellow.PULocationID == df_zones.LocationID) \
    .groupBy('Zone') \
    .count() \
    .orderBy('count', ascending=True) \
    .show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|       Rikers Island|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|New Dorp/Midland ...|   14|
|       West Brighton|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows
